In [1]:
pip install chromadb

  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached pybase64-1.5.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached onnxruntime-1.29.0-cp314-cp314-win_amd64.whl.metadata (5.8 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached grpcio-1.83.1-cp314-cp314-win_amd6


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
client

- *PyMuPDF:* 
    - Python Library to read and manipulate PDF files. 
    - It does text and image extraction in one library and is much faster on large docs than pypdf/pdfplumber.

- *sentence-transformers:* 
    - Turns text (and, with the right model, images) into embedding vectors —> fixed-length arrays of floats where similar meanings land close together. This is the piece that makes semantic search possible.
    - Example:
        - Text Model:
            ```
            model = SentenceTransformer("all-MiniLM-L6-v2")
            ```
        - Text + Images together ->  Use a CLIP(Contrastive Language-Image Pre-training) multimodal AI model or any other multimodal model:
            ```
            SentenceTransformer("clip-ViT-B-32") which embeds both images and text into the same vector space, so a text query can retrieve a relevant image. 
            ```

- *pillow:*
    - standard Python image library to open, convert, resize, crop, save images.

In [ ]:
pip install pymupdf sentence-transformers chromadb pillow

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.9/19.8 MB 16.4 MB/s eta 0:00:02
   ------------- -------------------------- 6.6/19.8 MB 16.9 MB/s eta 0:00:01
   --------------------- ------------------ 10.5/19.8 MB 18.4 MB/s eta 0:00:01
   ------------------------ --------------- 12.3/19.8 MB 15.5 MB/s eta 0:00:01
   -------------------------- ------------- 13.1/19.8 MB 14.0 MB/s eta 0:00:01
   ---------------------------- ----------- 13.9/19.8 MB 11.7 MB/s eta 0:00:01
   ------------------------------ --------- 15.2/19.8 MB 10.8 MB/s eta 0:00:01
   ------------------------------- -------- 15.7/19.8 MB 10.0 MB/s eta 0:00:01
   ------------------------------- -------- 15.7/19.8 MB 10.0 MB/s eta 0:00:01
   ------------------------------- -------- 15.7/19.8 MB 10.0 MB/s eta 0:00:01
   ------------------------------- -------- 15.7/19.8 MB 10.0 MB/s eta 0:00:01
   -------------------------------- ------- 16.0/19.8 MB 6.9 MB


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from typing import List, Dict, Any # used for type hinting in Python, allowing you to specify the expected types of variables, function parameters, and return values.

import fitz # used for importing the PyMuPDF library
from PIL import Image
import numpy as np
from sentence_transformers import SentenceTransformer # gonna help me out in downloading a model from Hugging Face and then using it.
import chromadb

In [8]:
from pathlib import Path

BASE_DIR = Path.cwd()                     # the 03_VectorDBs/ folder when run from VS Code / Jupyter
PDF_PATH = BASE_DIR / "attention_is_all_you_need.pdf"
FIGURES_DIR = BASE_DIR / "figures" # where the extracted images will be stored

FIGURES_DIR.mkdir(exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Put attention_is_all_you_need.pdf next to this notebook ({BASE_DIR})"
    )


In [9]:
import os
import chromadb
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # .env sits in the repo root, one level above this notebook

client = chromadb.CloudClient(
    api_key=os.environ["CHROMA_DB_API_KEY"],
    tenant=os.environ["CHROMA_DB_TENANT"],
    database=os.environ["CHROMA_DB_DATABASE"],
)

- The Simple strategy is to create two separate *"collections"* in my chromadb database(we have manually created the database).
    - one for textual data
    - one for images

In [10]:
FIG_COLLECTION_NAME = "attention_figures" 
TEXT_COLLECTION_NAME = "attention_text_chunks" 

In [ ]:
# We are using the "clip-ViT-B-32" model. The below code is taken from Hugging Face.

from sentence_transformers import SentenceTransformer, util # util is helper module that provides useful functions for common tasks like computing similarity scores between sentences or clustering texts

# Load CLIP model
clip_model = SentenceTransformer('clip-ViT-B-32')

c:\Users\AdityaRajPanda\OneDrive - Cubastion Consulting Pvt Ltd\Desktop\genai-bootcamp\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AdityaRajPanda\.cache\huggingface\hub\models--sentence-transformers--clip-ViT-B-32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|█████████

- The output embedding dimension of the clip-ViT-B-32 model is **512** which simply means the model produces 512-dimensional vectors.
- 512 defines the size of the representation space the model uses.
- *Dimension* here means the length (size) of the embedding vector that the model outputs.
- In models like CLIP, both images and text are converted into fixed-length numerical vectors(embeddings) that live in exactly the same mathematical space.
- Because they are in the same space, you can directly compare any image vector with any text vector using simple geometric measures such as cosine similarity or Euclidean distance.
- Embedding dimension = 512 means each output vector consists of exactly 512 numbers (usually floating-point values).
- So an embedding looks conceptually like:
```
[0.12, -0.45, 0.78, … , 0.03] ← 512 values in total
```
- Higher dimensions can in principle capture more nuanced information, but they also cost more memory and computation.

In [ ]:
# code for converting textual data into embeddings

def embed_text(text: str) -> np.ndarray:
    """
    Convert a given text into a vector embedding using the CLIP model.
    """
    # convert_to_numpy=True explicitly forces the method to return the final embeddings as a NumPy array
    # show_progress_bar is used to enable a progress bar in your interface or code. You can set it 'True' or 'False
    text_emb = clip_model.encode([text], convert_to_numpy=True, show_progress_bar=False)
    return text_emb[0] # extracts that first row from 2D array

In [ ]:
# code for converting image data into embeddings

def embed_image(path: str) -> np.ndarray:
    """
    Convert an image at the given path into a vector embedding using the CLIP model.
    """
    img = Image.open(path).convert("RGB")  # Open the image and convert it to RGB format
    img_emb = clip_model.encode([img], convert_to_numpy=True, show_progress_bar=True)
    return img_emb[0]

### Understanding 1D and 2D Arrays

#### 1D Array (One-Dimensional)
- **Structure:** A single row of values (like a line)
- **Shape:** `(n,)` where n = number of elements
- **Example:**
```python
  array_1d = [0.12, 0.56, 0.90, 0.34, 0.78]
  shape: (5,)  # 5 elements in ONE row
```
- **Use Case:** Represents a single embedding vector

---

#### 2D Array (Two-Dimensional)
- **Structure:** Multiple rows and columns (like a table/matrix)
- **Shape:** `(m, n)` where m = rows, n = columns
- **Example:**
```python
  array_2d = [[0.12, 0.56, 0.90, 0.34, 0.78],   # Row 0
              [0.15, 0.59, 0.91, 0.36, 0.75]]   # Row 1
  shape: (2, 5)  # 2 rows, 5 columns
```
- **Use Case:** Represents multiple embedding vectors (batch processing)

---

#### Accessing Elements

| Operation | Result | Type |
|-----------|--------|------|
| `array_2d[0]` | `[0.12, 0.56, 0.90, 0.34, 0.78]` | 1D array |
| `array_2d[1]` | `[0.15, 0.59, 0.91, 0.36, 0.75]` | 1D array |
| `array_2d[0, 2]` | `0.90` | Single value |

---

#### Why Shape is Always (1, 512) for CLIP-ViT-B-32?

##### Breaking Down (1, 512):
- **First dimension (1):** Number of texts you pass
  - You pass `[text]` → 1 text in the list → produces 1 row
  - If you passed `[text1, text2, text3]` → would be (3, 512)

- **Second dimension (512):** Embedding vector size of the model
  - `clip-ViT-B-32` always outputs **512-dimensional embeddings**
  - This is fixed by the model architecture
  - Every text gets converted to a vector of 512 numbers

##### Example:
```python
clip_model.encode([text])           # 1 text → shape: (1, 512)
clip_model.encode([text1, text2])   # 2 texts → shape: (2, 512)
clip_model.encode([text1, text2, text3, text4])  # 4 texts → shape: (4, 512)
```

The **512 never changes** (model-specific), but the **first number changes** based on how many texts you process.

---

#### Key Takeaway
**In the code:** `text_emb[0]` extracts the first row from a 2D array, converting it to a 1D embedding vector.
```python
text_emb = clip_model.encode([text])  # Returns 2D array: (1, 512)
                                      # 1 text, 512-dim embedding
embedding = text_emb[0]               # Extract row 0 → 1D array: (512,)
```


In [20]:
doc = fitz.open(PDF_PATH)

In [22]:
doc
doc.metadata

{'format': 'PDF 1.5',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'creator': 'LaTeX with hyperref',
 'producer': 'pdfTeX-1.40.25',
 'creationDate': 'D:20240410211143Z',
 'modDate': 'D:20240410211143Z',
 'trapped': '',
 'encryption': None}